In [1]:
import os

import scyjava
import imagej
import numpy as np
import xarray
import itertools

from omero import gateway
from getpass import getpass
from pprint import pprint

In [2]:
fiji_path = r"C:\Users\njg135\fiji-MetroloJ\Fiji"
omero_hostname = r"fms-fac-omero.ncl.ac.uk"
omero_username = "njg135"
temp_password = getpass("OMERO Password: ")

In [3]:
ij = imagej.init(fiji_path, mode="interactive")

Double = scyjava.jimport("java.lang.Double")

WindowManager = scyjava.jimport("ij.WindowManager")
Calibration = scyjava.jimport("ij.measure.Calibration")

MetroloJDialog = scyjava.jimport("metroloJ_QC.setup.MetroloJDialog")
QC_Options = scyjava.jimport("metroloJ_QC.setup.QC_Options")
simpleMetaData = scyjava.jimport("metroloJ_QC.importer.simpleMetaData")

coAlignement = scyjava.jimport("metroloJ_QC.coalignement.coAlignement")
coAlignementReport = scyjava.jimport("metroloJ_QC.report.coAlignementReport")

driftProfiler = scyjava.jimport("metroloJ_QC.stage.driftProfiler")
driftProfilerReport = scyjava.jimport("metroloJ_QC.report.driftProfilerReport")

PSFprofiler = scyjava.jimport("metroloJ_QC.resolution.PSFprofiler")
PSFprofilerReport = scyjava.jimport("metroloJ_QC.report.PSFprofilerReport")

In [4]:
def initialize_MetroloJDialog(method,
							  image,
							  thresholding_method="Otsu",
							  center_dectection_method="centroid",
							  save_pdf=False,
							  save_csv=False,
							  save_images=False):
	if method == "psf":
		method_string = "PSF profiler report generator"
	elif method == "drift":
		method_string = "Stage positioning and drift report generator"
	elif method == "registration":
		method_string = "Co-registration report generator"
	else:
		raise ValueError("Method must be one of 'psf', 'drift' or 'registration'")

	if thresholding_method not in ["Legacy", "Li", "Minimum", "Otsu"]:
		raise ValueError("Thresholding method must be one of 'Legacy', 'Li', 'Minimum', or 'Otsu'")
	
	if center_dectection_method == "ellipses":
		center_integer = 0
	elif center_dectection_method == "centroid":
		center_integer = 1
	elif center_dectection_method == "max":
		center_integer = 2
	else:
		raise ValueError("Center detection method must be one of 'ellipses', 'centroid', or 'max'")
	
	image_plus = image.generate_ImagePlus()

	WindowManager.setTempCurrentImage(image_plus)
	Dialog = MetroloJDialog(method_string, QC_Options())
	Dialog.beadDetectionThreshold = thresholding_method
	Dialog.centerDetectionMethodIndex = center_integer

	Dialog.savePdf = save_pdf
	Dialog.saveSpreadsheet = save_csv
	Dialog.saveImages = save_images

	Dialog.NA = image.NA
	Dialog.refractiveIndex = image.refractive_index
	Dialog.emWavelengths = [ch.emission_wave for ch in image.channels]
	Dialog.exWavelengths = [ch.excitation_wave for ch in image.channels]

	return Dialog

def execute_MetroloJ_process(Dialog, report_dir, report_name):
	image = Dialog.ip
	image_title = image.getTitle()
	creationInfo = simpleMetaData.getOMECreationInfos(image, Dialog.debugMode)
	coords = [Double.NaN, Double.NaN]
	
	if Dialog.reportType == "pp":
		execution_instance = PSFprofiler(image, Dialog, image_title, coords, creationInfo)
		report_instance = PSFprofilerReport(image, Dialog, image_title, coords, creationInfo)
	elif Dialog.reportType == "pos":
		execution_instance = driftProfiler(image, Dialog, image_title, coords, creationInfo)
		report_instance = driftProfilerReport(image, Dialog, image_title, coords, creationInfo)
	elif Dialog.reportType == "coa":
		execution_instance = coAlignement(image, Dialog, image_title, coords, creationInfo)
		report_instance = coAlignementReport(image, Dialog, image_title, coords, creationInfo)
	else:
		raise ValueError("Report types supported are PSF profiler, stage positioning and drift and co-registration")
	
	report_instance.saveReport(report_dir, report_name, None)
	
def connect(hostname, username, password):
    """
    Connect to an OMERO server
    :param hostname: Host name
    :param username: User
    :param password: Password
    :return: Connected BlitzGateway
    """
    conn = gateway.BlitzGateway(username, password,
                        host=hostname, secure=True, port=4063)
    conn.connect()
    conn.c.enableKeepAlive(60)
    return conn


def disconnect(conn):
    """
    Disconnect from an OMERO server
    :param conn: The BlitzGateway
    """
    conn.close()

class ChannelObject:
	def __init__(self, channel):
		self.channel = channel
		self.name = channel.getName()
		self.emission_wave = channel.getEmissionWave()
		self.excitation_wave = channel.getExcitationWave()
		self.mode = channel.getLogicalChannel().getMode().value
    
class ImageObject:
	def __init__(self, image, load_data=False):
		self.image = image
		self.id = image.getId()
		self.name = image.getName()
		self.size_x = image.getSizeX()
		self.size_y = image.getSizeY()
		self.size_z = image.getSizeZ()
		self.size_c = image.getSizeC()
		self.size_t = image.getSizeT()
		self.pixels = image.getPrimaryPixels()
		self.scale_x = self.pixels.getPhysicalSizeX()
		self.scale_y = self.pixels.getPhysicalSizeY()
		self.scale_z = self.pixels.getPhysicalSizeZ()
		self.dim_order = "TCZYX"
		self.objective = image.getObjectiveSettings()
		self.refractive_index = self.objective.getRefractiveIndex()
		self.NA = self.objective.getObjective().getLensNA()
		self.channels = [ChannelObject(ch) for ch in image.getChannels()]
		self.shape = (self.size_t, self.size_c, self.size_z, self.size_y, self.size_x)
		self.image_data = None
		if load_data:
			self.load_image_data()

	def load_plane(self, c, t, z):
		self.image_data[t, c, z, :, :] = np.array(self.pixels.getPlane(z, c, t))

	def load_image_data(self, c=None, t=None, z=None):
		if c is None:
			c = list(range(self.size_c))
		if t is None:
			t = list(range(self.size_t))
		if z is None:
			z = list(range(self.size_z))

		self.image_data = np.zeros((len(t), len(c), len(z), self.size_y, self.size_x))
		all_iterations = list(itertools.product(c, t, z))
		for args in all_iterations:
			self.load_plane(*args)
		self.image_data = xarray.DataArray(self.image_data, dims=["t", "ch", "pln", "row", "col"], name=self.name)
		self.shape = self.image_data.shape
	
	def generate_ImagePlus(self):
		if self.image_data is None:
			self.load_image_data()
		image_plus = ij.py.to_imageplus(self.image_data)
		CalibrationObj = Calibration()
		CalibrationObj.setXUnit(str(self.scale_x.getUnit()))
		CalibrationObj.setYUnit(str(self.scale_y.getUnit()))
		CalibrationObj.setZUnit(str(self.scale_z.getUnit()))
		CalibrationObj.pixelWidth = float(self.scale_x.getValue())
		CalibrationObj.pixelHeight = float(self.scale_y.getValue())
		CalibrationObj.pixelDepth = float(self.scale_z.getValue())
		image_plus.setCalibration(CalibrationObj)
		self.image_plus = image_plus
		return image_plus

In [21]:
# Python modules
import copy, re, os
from collections import Counter

Math = scyjava.jimport("java.lang.Math")
IJ = scyjava.jimport("ij.IJ")
WindowManager = scyjava.jimport("ij.WindowManager")
FileSaver = scyjava.jimport("ij.io.FileSaver")
Line = scyjava.jimport("ij.gui.Line")
Overlay = scyjava.jimport("ij.gui.Overlay")
Roi = scyjava.jimport("ij.gui.Roi")
Measurements = scyjava.jimport("ij.measure.Measurements")
ResultsTable = scyjava.jimport("ij.measure.ResultsTable")
ZProjector = scyjava.jimport("ij.plugin.ZProjector")
Analyzer = scyjava.jimport("ij.plugin.filter.Analyzer")

def analyzeParticles(
		Binary_Image, 
		Size_Setting, 
		Circularity_Setting):
	"""Runs analyze particles on the binary image, returning the ROI

	Args:
		Binary_Image (ij.ImagePlus): Segmented binary image
		Size_Setting (str): Min/Max size settings for analyse particles
		Circularity_Setting (str): Min/Max circularity settings for analyse particles

	Returns:
		[PolygonRoi]: Outputted Rois
	"""	

	# Defines analyse particles settings
	AnalyzeParticlesSettings = (
		"size=" 
		+ Size_Setting 
		+ " circularity=" 
		+ Circularity_Setting 
		+ " clear overlay exclude"
	)
	# Runs the analyze particles command to get ROI. 
	# Done by adding to the overlay in order to not have ROIManger shown to user
	IJ.run(Binary_Image, "Analyze Particles...", AnalyzeParticlesSettings)
	# Gets the Overlayed ROIs from analyze particles
	Overlayed_Rois = Binary_Image.getOverlay()
	# Takes the overlay and turns it into an array of ROI
	RoiList = Overlayed_Rois.toArray()
	# Removes this overlay to clean up the image
	IJ.run(Binary_Image, "Remove Overlay", "")
	return RoiList


def getRoiMeasurements(SampleRoi, Image, Measurement_Options):
	"""Gets the given measurements of the provided Roi for the given image

	Args:
		SampleRoi (ij.gui.Roi): Roi to be analysed
		Image (ij.ImagePlus): Image to be analysed
		Measurement_Options ([str]): ij.Measure.Measurements to be taken

	Returns:
		[float]: List of measurements in same order as Measurement_Options
	"""	
	
	# Initialises a new empty results table
	RTable = ResultsTable()
	# Initialises an Analyzer object using 
	# the image and the empty results table
	An = Analyzer(Image, RTable)
	# Selects the roi on the image
	Image.setRoi(SampleRoi)
	# Takes the measurements
	An.measure()
	# Takes the desired results from 
	# the results table and adds to a list
	OutputList = []
	for Option in Measurement_Options:
		OutputList.append(RTable.getValue(Option, 0))
	# Clears the results table
	RTable.reset()
	# Clears the roi from the image
	Image.resetRoi()
	return OutputList


def distanceBetweenPoints(X1, Y1, X2, Y2):
	"""Calculates the distance between two coordinates

	Args:
		X1 (float): Start X coordinate
		Y1 (float): Start Y coordinate
		X2 (float): End X coordinate
		Y2 (float): End Y coordinate

	Returns:
		float: Distance between the two points
	"""	
	xdiff = X1 - X2
	ydiff = Y1 - Y2
	Distance = Math.sqrt((xdiff*xdiff) + (ydiff*ydiff))
	return Distance


def closestPoint(Point, Point_List):
	"""Finds the closest two points in a list of ROIs

	Args:
		Roi (ij.gui.Roi): Roi to compare distances to
		RoiList (ij.gui.Roi[]): List of ROIs

	Returns:
		list: List of the two closest ROIs
	"""	
	MinDistance = float("Infinity")
	MinPoint = (None, None)

	for OtherPoint in Point_List:
		Distance = distanceBetweenPoints(Point[0], Point[1], OtherPoint[0], OtherPoint[1])
		if Distance < MinDistance:
			MinDistance = Distance
			MinPoint = OtherPoint
	return MinPoint
	

def roundToBase(Number, Base):
	"""Rounds the given number to the nearest multiple of the given base

	Args:
		Number (float): Number to be rounded
		Base (int): Base to round to

	Returns:
		int: Rounded number
	"""	
	RoundedNumber = (Base * Math.round(Number/Base))
	return RoundedNumber


def getAngleBetweenPoints(Point1, Point2):
	"""Gets the angle between two points in degrees

	Args:
		Point1 (tuple): X and Y coordinates of first point
		Point2 (tuple): X and Y coordinates of second point

	Returns:
		float: Angle between two points in degrees
	"""	
	Angle = Math.toDegrees(Math.atan2(Point2[1] - Point1[1], Point2[0] - Point1[0]))
	return Angle


def selectWindow(Pattern):
	"""Selects the window with the given pattern in the title

	Args:
		Pattern (string): regex pattern to match

	Returns:
		boolean: Whether the given window was found and selected
	"""	
	TitleList = WindowManager.getImageTitles()
	for Title in TitleList:
		Title = str(Title)
		if re.match(Pattern, Title):
			IJ.selectWindow(Title)
			return True
	return False

def run_z_accuracy(input_image,
		output_directory):
	# This section sets the measurements that will be used
	AnalyzerClass = Analyzer()
	# Gets original measurements to reset later
	OriginalMeasurements = AnalyzerClass.getMeasurements()

	# Sets the measurements to be used
	AnalyzerClass.setMeasurements(
		Measurements.SHAPE_DESCRIPTORS 
		+ Measurements.CENTROID
	)

	image_plus = input_image.generate_ImagePlus()

	# Gets the needed paths and filenames for input and output
	FileName = input_image.name
	FileNameNoExtension = ".".join(FileName.split("."))[:-1]
	OutputPath = output_directory
	#------------------------------------------------------^

	Calibration = image_plus.getCalibration()
	ZDepth = Calibration.pixelDepth

	# Max intensity of the image to get all of the ladder
	Projected = ZProjector.run(image_plus, "max")
	# Removes the scale so ROI coordinates are correct
	Projected.removeScale()
	# Thresholds the image to get the ladder
	IJ.setAutoThreshold(Projected, "Default dark")
	IJ.run(Projected, "Convert to Mask", "")
	# Runs analyze particles to get a list of ROIs
	RoiList = analyzeParticles(Projected, "10-Infinity", "0.00-1.00")

	# String needed to get the centroid of the ROI
	CentroidString = ["X", "Y"]

	# Gets the centroid of each ROI and adds to a list-------------------v
	PointList = []
	for ThisRoi in RoiList:
		Centroid = getRoiMeasurements(ThisRoi, Projected, CentroidString)
		# Must be a tuple to be hashable in dictionary
		PointList.append(tuple(Centroid))
	#--------------------------------------------------------------------^

	# For each point it will get the angle of the line between it and the closest point-v
	# This will be used to eliminate the points that are not part of the ladder
	# As these points will all be running parallel to each other
	PointDict = {}
	RoundedAngleList = []
	for Index, PointItem in enumerate(PointList):
		# Need to deep copy the list to avoid modifying the original
		InputList = copy.copy(PointList)
		# Need to remove the current point from the list to avoid finding itself
		del InputList[Index]
		# Finds the closest point to the current point
		ClosestPoint = closestPoint(PointItem, InputList)
		# Gets the angle between the two points
		LineAngle = getAngleBetweenPoints(PointItem, ClosestPoint)
		# Rounds the angle to the nearest 5 degrees
		# Has to be absolute as the lines can be in either direction
		RoundedAngle = abs(roundToBase(LineAngle, 5))
		if RoundedAngle >= 180:
			RoundedAngle -= 180
		# Adds the angle to a dictionary with the point as the key
		PointDict[PointItem] = RoundedAngle
		# Adds the angle to a list of all angles to find the mode
		RoundedAngleList.append(RoundedAngle)
	#-----------------------------------------------------------------------------------^

	# Gets the mode angle
	ModeAngle = Counter(RoundedAngleList).most_common(1)[0][0]
	# Gets the points that have the mode angle--------v
	# These are the points that are part of the ladder
	LadderList = []
	for PointItem in PointDict:
		if ModeAngle == PointDict[PointItem]:
			LadderList.append(PointItem)
	#-------------------------------------------------^

	# Gets the two points that are furthest apart but are still parallel to each other--------------------------v
	MaxLadderDistance = 0
	for Index, FirstPoint in enumerate(LadderList):
		# Need to deep copy the list to avoid modifying the original
		SecondList = copy.copy(LadderList)
		# Need to remove the current point from the list to avoid finding itself
		del SecondList[Index]
		# Loops though every other list of points to find the furthest apart
		for SecondPoint in SecondList:
			# Gets the angle between the two points, has to be absolute as the lines can be in either direction
			if FirstPoint[0] <= SecondPoint[0]:
				LadderAngle = getAngleBetweenPoints(FirstPoint, SecondPoint)
			else:
				LadderAngle = getAngleBetweenPoints(SecondPoint, FirstPoint)
			# Gets the distance between the two points
			LadderDistance = distanceBetweenPoints(FirstPoint[0], FirstPoint[1], SecondPoint[0], SecondPoint[1])
			# Rounds the angle to the nearest 5 degrees
			RoundedLadderAngle = abs(roundToBase(LadderAngle, 5)
)
			if RoundedLadderAngle >= 180:
				RoundedLadderAngle -= 180
			# If the angle is the same as the mode angle and the distance is greater than the current max
			# Then these are the new furthest apart points
			if RoundedLadderAngle == ModeAngle and LadderDistance > MaxLadderDistance:
				FeducialLine = Line(FirstPoint[0], FirstPoint[1], SecondPoint[0], SecondPoint[1])
				# This angle is not rounded as it is used to rotate the image
				FeducialAngle = LadderAngle
				MaxLadderDistance = LadderDistance
	#-----------------------------------------------------------------------------------------------------------^

	# Need to use an overlay so it will rotate with the image
	LineOverlay = Overlay(FeducialLine)
	image_plus.setOverlay(LineOverlay)
	
	# Rotates the image so the ladder is horizontal
	IJ.run(image_plus, "Arbitrarily...", "angle=" + str(-FeducialAngle) + " interpolate stack")
	# Gets the Rotated Roi from the overlay
	RotatedLineOverlay = image_plus.getOverlay()
	RotatedLineRoi = RotatedLineOverlay.get(0)
	
	# Removes the overlay to clean up the image
	image_plus.setOverlay(None)

	# Gets the centroid of the rotated line
	LineCentroid = getRoiMeasurements(RotatedLineRoi, Projected, CentroidString)

	# Gets the width of the image
	Width = image_plus.getWidth()
	# Creates a box roi that is 1 pixel high and the width of the image centred on the line centroid
	BoxRoi = Roi(0, LineCentroid[1], Width, 1)

	# Crops the image to the single line
	image_plus.setRoi(BoxRoi)
	LineImage = image_plus.crop("stack")
	# Closes the original image to save memory
	image_plus.close()

	# Runs the reslice command to get the XZ image similar to orthagonal view
	IJ.run(LineImage, "Reslice [/]...", "output=" + str(ZDepth) +" start=Top avoid")

	# Gets the resliced image
	selectWindow("Reslice ")
	OriginalSlicedImp = IJ.getImage()
	# Duplicates the image to only get one slice
	SlicedImp = OriginalSlicedImp.crop()
	# Close the original image to save memory
	OriginalSlicedImp.close()

	# Performs gaussian blur to smooth the image
	IJ.run(SlicedImp, "Gaussian Blur...", "sigma=6")
	# Gets the statistics which includes the minimum and maximum intensity of the image
	ImpStats = SlicedImp.getStatistics()
	# Sets the prominence for the find maxima command to be half the difference between the min and max intensity
	Prominence = str((ImpStats.max - ImpStats.min)/2)
	# Finds the maxima in the image and outputs to a results table
	IJ.run(SlicedImp, "Find Maxima...", "prominence=" + Prominence + " output=List")

	# Resets the contrast for easier viewing
	SlicedImp.resetDisplayRange()
	# Saves the XZ image and closes to save memory
	FileSaver(SlicedImp).saveAsTiff(os.path.join(OutputPath, FileNameNoExtension + "_XZ.tif"))
	SlicedImp.close()

	# Gets the results table and copies it so the displayed one can be closed
	Results = ResultsTable().getResultsTable()
	MaximaResults = Results.clone()
	# Needs to reset the table to avoid dialog asking to save
	Results.reset()
	# Closes the results table
	WindowManager.getWindow("Results").close()

	# Calculates the axial step size for each maxima
	for Row in range(0, MaximaResults.size()):
		AxialStep = MaximaResults.getValue("Y", Row) * ZDepth
		MaximaResults.setValue("AxialStep", Row, AxialStep)

	# Sorts the results table by the X coordinate
	MaximaResults.sort("X")

	# Calculates the axial difference between each maxima
	for SortedRow in range(1, MaximaResults.size()):
		AxialDiff = abs(MaximaResults.getValue("AxialStep", SortedRow) - MaximaResults.getValue("AxialStep", SortedRow - 1))
		MaximaResults.setValue("AxialDiff", SortedRow, AxialDiff)

	# Saves the results table
	MaximaResults.saveAs(os.path.join(OutputPath, FileNameNoExtension + "_XZ.csv"))

	# Resets the measurements to the original settings
	AnalyzerClass.setMeasurements(OriginalMeasurements)


In [17]:
connection = gateway.BlitzGateway(omero_username, temp_password, host=omero_hostname)
print ("Connecting to OMERO server...")
connection.connect()
print ("Connection successful!")

Connecting to OMERO server...
Connection successful!


In [18]:
zdrive_obj = ImageObject(connection.getObject("Image", 130324))

In [6]:
confocal_obj = ImageObject(connection.getObject("Image", 136561))
widefield_obj = ImageObject(connection.getObject("Image", 130658))
confocal_coreg = ImageObject(connection.getObject("Image", 130323))

In [19]:
zdrive_obj.load_image_data()

In [9]:
zdrive_imageplus = zdrive_obj.generate_ImagePlus()

In [22]:
run_z_accuracy(zdrive_obj, r"G:\QC\Ladder")

In [8]:
metro_dialog = initialize_MetroloJDialog("registration", confocal_coreg, save_pdf=True)
execute_MetroloJ_process(metro_dialog, r"G:\QC\\", "test_report_4")